# AK-Byte-1 — 100% Scratch Training\nThis notebook trains the custom 12-layer, 512-hidden, 8-head byte-level Transformer from random initialization. No Qwen, Llama, pretrained weights, or third-party model API is loaded. It resumes from Google Drive checkpoints and periodically saves progress. This is a small experimental model, not a ChatGPT-scale foundation model.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\nimport os, subprocess, torch\nREPO='/content/Ankit-Kryptex-'\nURL='https://github.com/sidharthkumar8580-oss/Ankit-Kryptex-.git'\nif not os.path.isdir(REPO): subprocess.run(['git','clone',URL,REPO],check=True)\nelse: subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)\nos.chdir(REPO)\nprint('PyTorch:',torch.__version__)\nprint('CUDA available:',torch.cuda.is_available())\nif torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0),'VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))\nelse: raise RuntimeError('GPU nahi mila. Colab > Runtime > Change runtime type > T4 GPU select karke dobara run karo.')

In [ ]:
!pip -q install -r requirements.txt

## Train / resume\nThe current repository has a small starter dataset. Training cannot create millions of genuine examples by repeating it. The run below performs up to 20,000 optimizer steps, validates periodically, and saves resumable checkpoints to Drive. If Colab disconnects, reconnect and rerun this cell; it resumes from the last saved checkpoint.

In [ ]:
SAVE='/content/drive/MyDrive/AK-Byte-1/weights.pt'\nos.makedirs(os.path.dirname(SAVE),exist_ok=True)\n!python train.py --steps 20000 --batch-size 4 --grad-accum 4 --lr 0.0002 --save-every 500 --eval-every 500 --save "$SAVE" --resume

In [ ]:
import torch\nfrom model.network import AKByteModel\nfrom model.config import BLOCK_SIZE\nck=torch.load(SAVE,map_location='cuda',weights_only=False)\nmodel=AKByteModel().cuda(); model.load_state_dict(ck['state_dict']); model.eval()\nprompt=('User: Hi\nAnkit Kryptex: ').encode('utf-8')\nx=torch.tensor(list(prompt),device='cuda',dtype=torch.long)[None,:]\nwith torch.no_grad(): out=model.generate(x,max_new_tokens=80)\nprint(bytes(out[0,len(prompt):].tolist()).decode('utf-8',errors='replace'))\nprint('Verified checkpoint step:',ck['step'],'best validation loss:',ck.get('best_val_loss'))

## Important\nThis training cell runs only while the Colab runtime is active. Free GPU availability and runtime duration are not guaranteed; Google says free sessions can run for at most 12 hours and may end earlier. Checkpoints are saved in Drive so training can resume. This notebook does not automatically deploy weights to Render.